# 02 - Connect models: finish setup

This is the final setup step. Check that ASSERT can reach the deployed models before starting an experiment. You do not need to choose a policy or design tests here.

Complete [deployment](./01_deploy_infra.ipynb) first and use **Python (assert01-foundry-evaluation)**.

## Load generated settings

Pick up the model connections saved during deployment. The table shows which model each label (`older_model`, `intermediate_model`, `newer_model`) refers to. Your Azure sign-in provides access; no keys belong in the notebook.

In [ ]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
from IPython.display import FileLink, display

demo_root = Path('..').resolve()
sys.path.insert(0, str(demo_root))
from app.assert_lab import artifact_path, load_connections, smoke_connections

metadata = load_connections(root=demo_root)
display(pd.DataFrame([model.to_dict() for model in metadata.deployments]))

## Connect models

Before designing tests, check that ASSERT can get both a text reply and a structured response from each model. Look for `READY` and `structured_ready = True` in the table.

**This calls the models and costs money:** two requests per model, plus any retries. It checks the connections, not whether the models follow your policy.

In [ ]:
smoke_results = await smoke_connections(metadata)
smoke_path = artifact_path('connections.json', root=demo_root)
smoke_path.write_text(json.dumps({
    'checked_at': datetime.now(timezone.utc).isoformat(),
    'subscription_id': metadata.subscription_id,
    'resource_group': metadata.resource_group,
    'api_version': metadata.api_version,
    'models': smoke_results,
    'skipped_deployment': metadata.skipped_deployment.to_dict() if metadata.skipped_deployment is not None else None,
    'skip_reason': metadata.skip_reason,
}, indent=2) + '\n', encoding='utf-8')
display(pd.DataFrame(smoke_results))
print(f'Non-secret connection evidence: {smoke_path}')
display(FileLink((Path('..') / smoke_path.relative_to(demo_root)).as_posix()))
if metadata.skipped_deployment is not None:
    print(f'Partial verification only: {metadata.skip_reason}')

## Setup complete - start your ASSERT experiment

Once the connection checks pass, your environment is ready. You can reuse this setup for different policies without redeploying the models.

Open [03_assert_flow.ipynb](./03_assert_flow.ipynb) to begin the ASSERT flow: choose a policy, configure the experiment, review tests, evaluate, and compare. From here on, the focus is the assistant's behavior rather than Azure setup.